In [ ]:
# 필요한 패키지 설치 (Colab 첫 셀에서 실행)
# !pip install -q langchain langchain-community pypdf

# === Colab에 세 가지 파일 동시 업로드 ===
from google.colab import files
uploaded = files.upload()
# 파일 선택 다이얼로그에서
# sample_textbook.md, sample_homepage.html, sample_pdf.pdf를 한 번에 선택

print(f"업로드된 파일: {list(uploaded.keys())}")

In [ ]:
import os
import shutil

# === 폴더 구조 생성 ===
os.makedirs("docs/text", exist_ok=True)   # .md, .html용
os.makedirs("docs/pdf", exist_ok=True)    # .pdf용

# === 파일을 확장자별로 적절한 폴더로 이동 ===
shutil.move("sample_textbook.md",   "docs/text/sample_textbook.md")
shutil.move("sample_homepage.html", "docs/text/sample_homepage.html")
shutil.move("sample_pdf.pdf",       "docs/pdf/sample_pdf.pdf")

# === 구성된 폴더 구조 확인 ===
for root, dirs, fnames in os.walk("docs"):
    for f in fnames:
        print(os.path.join(root, f))

In [ ]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader

# === docs/text/ 아래의 .md 파일과 .html 파일을 각각 로딩 ===
md_loader = DirectoryLoader(
    path="docs/text",
    glob="**/*.md",
    loader_cls=TextLoader,
    loader_kwargs={"encoding": "utf-8"},
)

html_loader = DirectoryLoader(
    path="docs/text",
    glob="**/*.html",
    loader_cls=TextLoader,
    loader_kwargs={"encoding": "utf-8"},
)

md_docs = md_loader.load()
html_docs = html_loader.load()

print(f".md Document 수: {len(md_docs)}")
print(f".html Document 수: {len(html_docs)}\n")

for d in md_docs + html_docs:
    print(f"- [{d.metadata['source']}] ({len(d.page_content)}자)")

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

# === docs/pdf/ 아래의 .pdf 파일을 로딩 ===
pdf_loader = DirectoryLoader(
    path="docs/pdf",
    glob="**/*.pdf",
    loader_cls=PyPDFLoader,
)

pdf_docs = pdf_loader.load()

print(f".pdf Document 수: {len(pdf_docs)}\n")
for d in pdf_docs:
    print(f"- [{d.metadata['source']}] page={d.metadata['page']} ({len(d.page_content)}자)")

In [ ]:
# === 세 로더의 결과를 하나의 Document 리스트로 합침 ===
all_docs = md_docs + html_docs + pdf_docs

print(f"전체 Document 수: {len(all_docs)}\n")
for d in all_docs:
    src = d.metadata['source']
    page = d.metadata.get('page')
    suffix = f" (page={page})" if page is not None else ""
    print(f"- {src}{suffix} | {len(d.page_content)}자")

In [ ]:
loader = DirectoryLoader(
    path="docs/text",
    glob="**/*.md",
    loader_cls=TextLoader,
    loader_kwargs={"encoding": "utf-8"},
    show_progress=True,            # 진행률 표시
    use_multithreading=True,       # 병렬 처리
)

docs = loader.load()
print(f"\n총 {len(docs)}개의 Document를 로딩했습니다.")